# Lezione 11 — Traduzione vocale: il progetto Babel Fish

## Obiettivi di apprendimento

- Capire l'architettura STT → LLM → TTS applicata alla traduzione (progetto Babel Fish di IBM).
- Individuare perché il prompt "traduttore" originale del materiale fonte non traduce davvero.
- Costruire un mini-traduttore vocale 100% gratuito con Hugging Face (`Helsinki-NLP/opus-mt-en-es`) e, facoltativamente, dargli voce con `gTTS`.
- Confrontare la strada "LLM generalista + prompt" con la strada "modello specializzato in traduzione".

## Cosa costruiremo

Un traduttore da inglese a spagnolo che riceve testo (in un progetto reale, arriverebbe da Whisper che trascrive l'audio), lo traduce con un modello gratuito di Hugging Face, e facoltativamente legge la traduzione ad alta voce.

## Ripasso teorico: l'architettura Babel Fish

Il progetto **Babel Fish** di IBM (`BabelFish.pdf`, progetto `babel-fish-project/translator-with-voice-and-watsonx/`) riutilizza la stessa architettura dell'assistente vocale conversazionale:

```
[Browser] --registra audio, POST--> [Flask server.py]
  POST /speech-to-text   -> Watson STT   -> testo
  POST /process-message  -> watsonx (Mistral) -> traduzione -> Watson TTS -> audio
```

Tre funzioni in `worker.py`: `speech_to_text`, `watsonx_process_message` (l'LLM) e `text_to_speech`.

**La scoperta reale**: nell'implementazione che circola del progetto, `watsonx_process_message` usa questo prompt generico:

```python
def watsonx_process_message(user_message):
    prompt = f"""Respond to the query: ```{user_message}```"""
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

Questo **non** traduce nulla — è un prompt di "rispondi a questo", non di "traduci questo". Lo stesso PDF documenta il prompt corretto (specificando lingua di origine e di destinazione), ma l'implementazione reale non lo usa. È un bug reale del materiale fonte, e la lezione è: **testa i tuoi prompt, non dare per scontato che facciano quello che il loro nome promette.**

Qui sotto trovi quel codice reale di watsonx come riferimento — non lo eseguiremo (richiede credenziali a pagamento). Al suo posto, costruiamo la stessa idea con un'alternativa 100% gratuita.

### Blocco di riferimento — codice reale di watsonx (NON eseguibile senza credenziali IBM)

Così si fa con le API a pagamento usate dal materiale ufficiale di IBM. Qui sotto usiamo l'alternativa gratuita di Hugging Face per poter seguire la lezione senza costi.

```python
from ibm_watson_machine_learning.foundation_models import Model

# Prompt CORRETTO documentato nel PDF (ma non usato nell'implementazione reale):
def watsonx_process_message(user_message):
    prompt = f"""
    Translate the following English sentence into Spanish.
    Reply ONLY with the translation, no explanations, no formatting, no extra text.
    English: {user_message}
    Spanish:
    """
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()

# Prompt REALE trovato nel progetto (bug: non traduce davvero)
def watsonx_process_message_bug(user_message):
    prompt = f"""Respond to the query: ```{user_message}```"""
    response_text = model.generate_text(prompt=prompt)
    return response_text.strip()
```

In [ ]:
# Installiamo le librerie che useremo (gratuite, senza API key).
# transformers + sentencepiece: per il modello di traduzione di Hugging Face.
# gTTS: per convertire la traduzione in audio (Google Text-to-Speech, gratuito).
!pip install -q transformers sentencepiece gTTS

## Passo 1 — Caricare un modello di traduzione reale e gratuito

`Helsinki-NLP/opus-mt-en-es` è un modello **specializzato in traduzione** inglese→spagnolo (non un LLM generalista che bisogna "convincere" con un prompt). Questa è la differenza chiave rispetto al bug di Babel Fish: questo modello sa solo tradurre, quindi non può "dimenticarsi" di farlo.

In [ ]:
from transformers import pipeline

# Carichiamo una pipeline di traduzione gia pronta: scarica il modello la prima volta
# e poi lo riutilizza. Non serve nessuna chiave API.
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")

# Proviamo con una frase di esempio
resultado = translator("Where is the nearest train station?")
print(resultado[0]["translation_text"])

## Passo 2 — Riprodurre il bug di Babel Fish per vederlo con i tuoi occhi

Simuliamo il prompt generico del bug reale (`"Respond to the query: ..."`) usando lo stesso modello di traduzione come se fosse un LLM a cui "chiediamo" qualsiasi cosa con un prompt di testo semplice. Dato che `Helsinki-NLP/opus-mt-en-es` è un modello *seq2seq* specializzato, non accetta istruzioni in linguaggio naturale come un LLM — quindi per illustrare davvero il bug, confrontiamo concettualmente le due funzioni `watsonx_process_message` della cella di riferimento qui sopra.

Il confronto reale che conta è questa tabella mentale:

| Funzione | Prompt | Traduce davvero? |
|---|---|---|
| `watsonx_process_message` (PDF, corretta) | chiede esplicitamente di tradurre EN→ES | Sì |
| `watsonx_process_message_bug` (progetto reale) | "rispondi a questo" generico | Non necessariamente |
| `translator(...)` (Hugging Face, questa lezione) | non serve alcun prompt, il modello traduce e basta | Sì, per progettazione |

In [ ]:
# Verifichiamo il traduttore con diverse frasi, incluse domande,
# ordini e frasi lunghe, per controllare che traduca SEMPRE (non solo a volte).
frases = [
    "Good morning, how can I help you?",
    "The flight to Madrid has been delayed by two hours.",
    "Please, translate this sentence carefully.",
]

for frase in frases:
    traduccion = translator(frase)[0]["translation_text"]
    print(f"EN: {frase}")
    print(f"ES: {traduccion}")
    print("---")

## Passo 3 — Dare voce alla traduzione (facoltativo, con gTTS)

`gTTS` (Google Text-to-Speech) converte il testo in un file audio, gratuitamente. È l'equivalente semplice del Watson TTS usato da Babel Fish nell'architettura reale.

In [ ]:
from gtts import gTTS
from IPython.display import Audio

texto_original = "The flight to Madrid has been delayed by two hours."
traduccion = translator(texto_original)[0]["translation_text"]
print("Traduzione:", traduccion)

# Generiamo l'audio in spagnolo a partire dalla traduzione (non dal testo originale).
tts = gTTS(text=traduccion, lang="es")
tts.save("traduccion.mp3")

# Riproduciamo l'audio direttamente nel notebook.
Audio("traduccion.mp3")

## Riepilogo della lezione

- Babel Fish riutilizza l'architettura STT → LLM → TTS dell'assistente vocale, cambiando il compito dell'LLM in "tradurre".
- Il materiale fonte ha un bug reale: il prompt usato nell'implementazione non forza la traduzione, nonostante si chiami "traduttore".
- Un modello specializzato in traduzione (`Helsinki-NLP/opus-mt-en-es`) traduce per progettazione, senza dipendere dal fatto che un prompt sia scritto bene.
- `gTTS` ti permette di aggiungere voce a qualsiasi testo gratuitamente, chiudendo il ciclo testo → voce.

**Continua con:**
- Capitolo del manuale: [15_traduccion_babelfish.md](../manual/15_traduccion_babelfish_IT.md)
- Notebook di pratica: [clase11_practica_traduccion_babelfish.ipynb](../notebooks_practica/clase11_practica_traduccion_babelfish_IT.ipynb)